# Capstone Project

##  PERSONALIZED VIRTUAL SHOPPING ASSISTANT ON WHATSAPP USING LLAMA STACK

### Preparing Recommendation Engine

## Dataset

The columns provide information about the transactions, customers, products, and purchasing behavior, making the dataset suitable for various analyses, including market basket analysis and customer segmentation. Here's a brief explanation of each column in the Dataset:

  Transaction_ID: A unique identifier for each transaction, represented as a 10-digit number. This column is used to uniquely identify each purchase.

  Date: The date and time when the transaction occurred. It records the timestamp of each purchase.

  Customer_Name: The name of the customer who made the purchase. It provides information about the customer's identity.

  Product: A list of products purchased in the transaction. It includes the names of the products bought.

  Total_Items: The total number of items purchased in the transaction. It represents the quantity of products bought.

  Total_Cost: The total cost of the purchase, in currency. It represents the financial value of the transaction.

  Payment_Method: The method used for payment in the transaction, such as credit card, debit card, cash, or mobile payment.

  City: The city where the purchase took place. It indicates the location of the transaction.

  Store_Type: The type of store where the purchase was made, such as a supermarket, convenience store, department store, etc.

  Discount_Applied: A binary indicator (True/False) representing whether a discount was applied to the transaction.

  Customer_Category: A category representing the customer's background or age group.

  Season: The season in which the purchase occurred, such as spring, summer, fall, or winter.
  
  Promotion: The type of promotion applied to the transaction, such as "None," "BOGO (Buy One Get One)," or "Discount on Selected Items."



#### Import required packages

In [1]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import os

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

### Load the data


In [3]:
df=pd.read_csv('/content/Retail_Transactions_Dataset.csv')

In [4]:
df.head(2)

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000000,21-01-2022 06:27,Stacey Price,"['Ketchup', 'Shaving Cream', 'Light Bulbs']",3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN
1,1000000001,01-03-2023 13:01,Michelle Carlson,"['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'P...",2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One)


In [5]:
df.shape

(1000000, 13)

In [6]:
df.dtypes

,0
Transaction_ID,int64
Date,object
Customer_Name,object
Product,object
Total_Items,int64
Total_Cost,float64
Payment_Method,object
City,object
Store_Type,object
Discount_Applied,bool


In [7]:
# Flatten the List of Product
# Convert the 'Product' column (which should be a string representation of a list) into actual lists
df['Product'] = df['Product'].apply(lambda x: eval(x) if isinstance(x, str) else x)

In [8]:
# Extracting All Unique Items
# Flatten the list of items into one big list
all_items = [Product for sublist in df['Product'] for Product in sublist]

# Get unique items
unique_items = set(all_items)

# Convert to a sorted list (optional)
unique_items = sorted(list(unique_items))

# Display the unique items
print(unique_items)


['Air Freshener', 'Apple', 'BBQ Sauce', 'Baby Wipes', 'Banana', 'Bath Towels', 'Beef', 'Bread', 'Broom', 'Butter', 'Canned Soup', 'Carrots', 'Cereal', 'Cereal Bars', 'Cheese', 'Chicken', 'Chips', 'Cleaning Rags', 'Cleaning Spray', 'Coffee', 'Deodorant', 'Diapers', 'Dish Soap', 'Dishware', 'Dustpan', 'Eggs', 'Extension Cords', 'Feminine Hygiene Products', 'Garden Hose', 'Hair Gel', 'Hand Sanitizer', 'Honey', 'Ice Cream', 'Insect Repellent', 'Iron', 'Ironing Board', 'Jam', 'Ketchup', 'Laundry Detergent', 'Lawn Mower', 'Light Bulbs', 'Mayonnaise', 'Milk', 'Mop', 'Mustard', 'Olive Oil', 'Onions', 'Orange', 'Pancake Mix', 'Paper Towels', 'Pasta', 'Peanut Butter', 'Pickles', 'Plant Fertilizer', 'Potatoes', 'Power Strips', 'Razors', 'Rice', 'Salmon', 'Shampoo', 'Shaving Cream', 'Shower Gel', 'Shrimp', 'Soap', 'Soda', 'Spinach', 'Sponges', 'Syrup', 'Tea', 'Tissues', 'Toilet Paper', 'Tomatoes', 'Toothbrush', 'Toothpaste', 'Trash Bags', 'Trash Cans', 'Tuna', 'Vacuum Cleaner', 'Vinegar', 'Water',

In [9]:
# Count the number of unique items
num_unique_items = len(unique_items)

# Print the count of unique items
print(f"Number of unique items: {num_unique_items}")

Number of unique items: 81


## Data Preprocessing

In [10]:
# We will first explode the product list so that each row represents a customer-product interaction.
df_exploded = df.explode('Product')
df_exploded.head()

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000000,21-01-2022 06:27,Stacey Price,Ketchup,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN
0,1000000000,21-01-2022 06:27,Stacey Price,Shaving Cream,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN
0,1000000000,21-01-2022 06:27,Stacey Price,Light Bulbs,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN
1,1000000001,01-03-2023 13:01,Michelle Carlson,Ice Cream,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One)
1,1000000001,01-03-2023 13:01,Michelle Carlson,Milk,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One)


In [11]:
# Create a user-item matrix; used aagfunc=count as duplicate present in the total items column
interaction_matrix = df_exploded.pivot_table(index='Customer_Name', columns='Product', values='Total_Items', aggfunc='count').fillna(0)

interaction_matrix.head()

Product,Air Freshener,Apple,BBQ Sauce,Baby Wipes,Banana,Bath Towels,Beef,Bread,Broom,Butter,...,Tomatoes,Toothbrush,Toothpaste,Trash Bags,Trash Cans,Tuna,Vacuum Cleaner,Vinegar,Water,Yogurt
Customer_Name,,,,,,,,,,,,,,,,,,,,,
Aaron Acevedo,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Aaron Acosta,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
Aaron Adams,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
Aaron Adkins,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Aaron Aguilar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
interaction_matrix.shape

(329738, 81)

In [13]:
from sklearn.metrics.pairwise import cosine_similarity
# similarity_matrix = cosine_similarity(interaction_matrix)  # User-based filtering

similarity_matrix = cosine_similarity(interaction_matrix.T)  # Item-based filtering

In [15]:
user_index = 0  # Index of a user
user_interactions = interaction_matrix.iloc[user_index]
similar_items = similarity_matrix.dot(user_interactions)
recommended_items = similar_items.argsort()[-5:][::-1]  # Top 5 recommendations

In [16]:
print(recommended_items)

[47 73 44  2 75]


In [17]:
recommended_product_names = interaction_matrix.columns[recommended_items].tolist()

print("Recommended Products:", recommended_product_names)

Recommended Products: ['Orange', 'Toothpaste', 'Mustard', 'BBQ Sauce', 'Trash Cans']


**Apply the Collaborative Filtering Method:**

**Generate Recommendations:**

In [18]:
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

# Prepare a user-item interaction matrix (replace with your actual data)
interaction_matrix = df_exploded.pivot_table(index='Customer_Name', columns='Product', values='Total_Items', aggfunc='count').fillna(0)

# Apply SVD for Collaborative Filtering
svd = TruncatedSVD(n_components=50, random_state=42)
svd_matrix = svd.fit_transform(interaction_matrix)

# Reconstruct the matrix to make predictions
reconstructed_matrix = svd.inverse_transform(svd_matrix)

# Convert the reconstructed matrix back to a DataFrame for easy access
reconstructed_df = pd.DataFrame(reconstructed_matrix, columns=interaction_matrix.columns, index=interaction_matrix.index)

In [22]:
# Generate recommendations for a user (example: customer 'Stacey Price')
user_recommendations = reconstructed_df.loc['Aaron Acevedo'].sort_values(ascending=False)

# Show top 5 recommended products for the user
print("Collaborative Filtering Recommendations for 'Aaron Acevedo':")
print(user_recommendations.head(10))

Collaborative Filtering Recommendations for 'Aaron Acevedo':
Product
Orange            1.268225
Sponges           0.861089
Mustard           0.839498
Trash Cans        0.721193
Hand Sanitizer    0.655491
Diapers           0.625097
Potatoes          0.622765
Shrimp            0.544672
BBQ Sauce         0.537674
Hair Gel          0.367803
Name: Aaron Acevedo, dtype: float64


**Filter out the products if already interacted**

In [23]:
# Get products Stacey Price has already interacted with
already_purchased = interaction_matrix.loc['Aaron Acevedo']
already_purchased = already_purchased[already_purchased > 0].index

# Filter out already purchased items from recommendations
filtered_recommendations = user_recommendations.drop(labels=already_purchased)

# Show top 5 new recommendations
print("Top 5 NEW Recommendations for 'Aaron Acevedo':")
print(filtered_recommendations.head(5))

Top 5 NEW Recommendations for 'Aaron Acevedo':
Product
Shrimp       0.544672
Hair Gel     0.367803
Spinach      0.356438
Razors       0.348909
Olive Oil    0.302032
Name: Aaron Acevedo, dtype: float64


In [24]:
print("Recommended Products:")
for product, score in filtered_recommendations.head(5).items():
    print(f"{product}: {score:.2f}")

Recommended Products:
Shrimp: 0.54
Hair Gel: 0.37
Spinach: 0.36
Razors: 0.35
Olive Oil: 0.30


**Prediction Accuracy checking**

In [25]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Flatten matrices
actual = interaction_matrix.values.flatten()
predicted = reconstructed_df.values.flatten()

# RMSE
rmse = np.sqrt(mean_squared_error(actual, predicted))
print(f"RMSE: {rmse:.4f}")

# MAE
mae = mean_absolute_error(actual, predicted)
print(f"MAE: {mae:.4f}")

RMSE: 0.2039
MAE: 0.1298


**Generate Final Recommendations:**

Market Basket Analysis: Based on the rules generated by Apriori/FP-Growth, recommend products based on the association rules (i.e., "If you bought X, you might like Y").

Collaborative Filtering: Based on matrix factorization (SVD), generate a list of predicted items for each user.

Market Basket Analysis works best for transactional data and is great for finding item associations.

Collaborative Filtering (SVD) works well when you have a large user-item interaction matrix and can generate personalized recommendations.

RMSE (for Collaborative Filtering):

To measure the accuracy of the Collaborative Filtering model, we can compute the Root Mean Squared Error (RMSE) between the predicted ratings and the actual interactions.

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Example: Create a test set with actual interactions for the user (for evaluation)
# Assuming 'interaction_matrix' is the user-item matrix where entries are ratings or interaction frequencies
actual_values = interaction_matrix.loc['Stacey Price']
predicted_values = reconstructed_df.loc['Stacey Price']

# RMSE for Collaborative Filtering (SVD)
rmse = np.sqrt(mean_squared_error(actual_values, predicted_values))
print(f"Collaborative Filtering RMSE: {rmse:.2f}")